<p class="h1">ECE 447 - Notebook 11</p>
<p class="h2">KNNs and Clustering</p>

credit: some contents are taken from https://dashee87.github.io/data%20science/general/Clustering-with-Scikit-with-GIFs/

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_blobs, make_moons, make_circles
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neighbors import NearestCentroid
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.metrics import adjusted_rand_score

# KNNs

In [ ]:
def make_anisotropic_blobs(n=100, random_state=0):
    X, y = make_blobs(n_samples=n, n_features=2, centers=2, cluster_std=1.2, random_state=random_state)
    rng = np.random.RandomState(random_state)
    A = np.array([[0.6, -0.3],
                  [0.2,  0.7]])
    X = X @ A
    return X, y

def make_unequal_variance_blobs(n=100, random_state=0):
    X, y = make_blobs(
        n_samples=n, centers=[(-2, 0), (2, 0)],
        cluster_std=[0.6, 2.2], random_state=random_state
    )
    return X, y

def make_imbalanced_moons(n=100, random_state=0):
    X, y = make_moons(n_samples=n, noise=0.25, random_state=random_state)
    # downsample class 1 to create imbalance
    rng = np.random.RandomState(random_state)
    idx0 = np.where(y == 0)[0]
    idx1 = np.where(y == 1)[0]
    keep1 = rng.choice(idx1, size=int(0.35 * len(idx1)), replace=False)
    idx = np.concatenate([idx0, keep1])
    rng.shuffle(idx)
    return X[idx], y[idx]

In [ ]:
def plot_decision_boundary(ax, clf, X, y, title="", h=0.03, pad=0.6):
    x_min, x_max = X[:, 0].min() - pad, X[:, 0].max() + pad
    y_min, y_max = X[:, 1].min() - pad, X[:, 1].max() + pad
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    grid = np.c_[xx.ravel(), yy.ravel()]
    Z = clf.predict(grid).reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.25)
    ax.scatter(X[:, 0], X[:, 1], c=y, s=18, edgecolor="k", linewidth=0.2)
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])

In [ ]:
n_samples = 600
k = 9
test_size = 0.3

datasets = [
    ("Linearly separable blobs", lambda: make_blobs(n_samples=n_samples, centers=2, cluster_std=1.0, random_state=0)),
    ("Anisotropic blobs",        lambda: make_anisotropic_blobs(n=n_samples, random_state=0)),
    ("Unequal variance blobs",   lambda: make_unequal_variance_blobs(n=n_samples, random_state=0)),
    ("Moons (nonlinear)",        lambda: make_moons(n_samples=n_samples, noise=0.25, random_state=0)),
    ("Circles (nonlinear)",      lambda: make_circles(n_samples=n_samples, noise=0.08, factor=0.5, random_state=0)),
    ("Imbalanced moons",         lambda: make_imbalanced_moons(n=n_samples, random_state=0)),
]

models = [
    ("MED (Nearest Class Mean)", make_pipeline(StandardScaler(), NearestCentroid())),
    ("1-NN",                     make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=1))),
    ("kNN (k=9)",                make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))),
]

fig, axes = plt.subplots(len(datasets), len(models), figsize=(4.0*len(models), 3.3*len(datasets)))
if len(datasets) == 1:
    axes = np.array([axes])

for i, (dname, dgen) in enumerate(datasets):
    X, y = dgen()
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=0, stratify=y)

    for j, (mname, clf) in enumerate(models):
        clf.fit(X_train, y_train)
        yhat = clf.predict(X_test)
        acc = accuracy_score(y_test, yhat)

        ax = axes[i, j]
        plot_decision_boundary(ax, clf, X_test, y_test, title=f"{mname}\nacc={acc:.3f}")
        if j == 0:
            ax.set_ylabel(dname, rotation=90, labelpad=22, fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
X, y = make_circles(n_samples=700, noise=0.10, factor=0.45, random_state=1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.35, random_state=0, stratify=y)

clfs = [
    ("MED", make_pipeline(StandardScaler(), NearestCentroid())),
    ("1-NN", make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=1))),
    ("kNN (k=15)", make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15))),
]

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
for ax, (name, clf) in zip(axes, clfs):
    clf.fit(X_train, y_train)
    acc = accuracy_score(y_test, clf.predict(X_test))
    plot_decision_boundary(ax, clf, X_test, y_test, title=f"{name}  acc={acc:.3f}")
plt.tight_layout()
plt.show()

In [ ]:
X, y = make_moons(n_samples=700, noise=0.30, random_state=0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.35, random_state=0, stratify=y)

ks = [1, 3, 9, 25, 75, 200]
fig, axes = plt.subplots(1, len(ks), figsize=(3.4*len(ks), 3.2))

for ax, k in zip(axes, ks):
    print("k is", k)
    clf = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)
    acc = accuracy_score(y_test, pred)
    plot_decision_boundary(ax, clf, X_test, y_test, title=f"k={k}\nacc={acc:.3f}")

    print(confusion_matrix(y_test,pred))
    print(classification_report(y_test,pred))

plt.tight_layout()
plt.show()

In [ ]:

X, y = make_moons(n_samples=800, noise=0.30, random_state=0)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=0, stratify=y
)

# ----- k range -----
ks = list(range(1, 76, 2))  # odd k's from 1 to 75

# ----- Storage -----
acc_te, prec_te, rec_te, f1_te = [], [], [], []
acc_tr, prec_tr, rec_tr, f1_tr = [], [], [], []

for k in ks:
    clf = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
    clf.fit(X_train, y_train)

    yhat_test = clf.predict(X_test)
    yhat_train = clf.predict(X_train)

    # test metrics
    acc_te.append(accuracy_score(y_test, yhat_test))
    prec_te.append(precision_score(y_test, yhat_test, zero_division=0))
    rec_te.append(recall_score(y_test, yhat_test, zero_division=0))

    # train metrics (optional but very instructive)
    acc_tr.append(accuracy_score(y_train, yhat_train))
    prec_tr.append(precision_score(y_train, yhat_train, zero_division=0))
    rec_tr.append(recall_score(y_train, yhat_train, zero_division=0))

# ----- Plot -----
plt.figure(figsize=(8, 5))
plt.plot(ks, acc_te, label="Accuracy (test)")
plt.plot(ks, prec_te, label="Precision (test)")
plt.plot(ks, rec_te, label="Recall (test)")

# optional: show train curves (comment out if too busy visually)
plt.plot(ks, acc_tr, linestyle="--", label="Accuracy (train)")

plt.xlabel("k (number of neighbors)")
plt.ylabel("Metric value")
plt.title("kNN performance vs. k")
# plt.ylim(0, 1.05)
plt.grid(True, alpha=0.25)
plt.legend()
plt.show()

In [ ]:
# several clusters (more than 2 example)

n_samples = 9
k = 9
test_size = 0.3

datasets = [
    ("Linearly separable blobs", lambda: make_blobs(n_samples=n_samples, centers=9, cluster_std=1.0, random_state=287)),
]

models = [
    ("MED (Nearest Class Mean)", make_pipeline(StandardScaler(), NearestCentroid())),
    ("1-NN",                     make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=1))),
    ("kNN (k=9)",                make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))),
]

fig, axes = plt.subplots(len(datasets), len(models), figsize=(4.0*len(models), 3.3*len(datasets)))
if len(datasets) == 1:
    axes = np.array([axes])

for i, (dname, dgen) in enumerate(datasets):
    X, y = dgen()

    for j, (mname, clf) in enumerate(models):
        clf.fit(X, y)
        yhat = clf.predict(X)
        acc = accuracy_score(y, yhat)

        ax = axes[i, j]
        plot_decision_boundary(ax, clf, X, y, title=f"{mname}\nacc={acc:.3f}")
        if j == 0:
            ax.set_ylabel(dname, rotation=90, labelpad=22, fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
n_samples = 300
n_neighbors = 6
a = 100.0

# Create a dataset where feature scales differ
X, y = make_blobs(n_samples=n_samples, centers=2, cluster_std=1.2, random_state=0)
X[:, 0] *= a  # blow up x1 scale

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.35, random_state=0, stratify=y)

clfs = [
    ("kNN (no scaling)", KNeighborsClassifier(n_neighbors=n_neighbors)),
    ("kNN (with scaling)", make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=n_neighbors))),
]

fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
for ax, (name, clf) in zip(axes, clfs):
    clf.fit(Xtr, ytr)
    acc = accuracy_score(yte, clf.predict(Xte))
    plot_decision_boundary(ax, clf, Xte, yte, title=f"{name}\nacc={acc:.3f}")
plt.tight_layout()
plt.show()

In [ ]:
n_samples = 300
n_neighbors = 6
a = 20.0

# Create a dataset where feature scales differ
X, y = make_blobs(n_samples=n_samples, centers=2, cluster_std=1.2, random_state=0)
X[:, 0] *= a  # blow up x1 scale

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.35, random_state=0, stratify=y)

clfs = [
    ("kNN (no scaling)", KNeighborsClassifier(n_neighbors=n_neighbors)),
    ("kNN (with scaling)", make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=n_neighbors))),
]

fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
for ax, (name, clf) in zip(axes, clfs):
    clf.fit(Xtr, ytr)
    acc = accuracy_score(yte, clf.predict(Xte))
    plot_decision_boundary(ax, clf, Xte, yte, title=f"{name}\nacc={acc:.3f}")
plt.tight_layout()
plt.show()

# Clustering

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_blobs, make_moons, make_circles
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

from sklearn.cluster import KMeans, AgglomerativeClustering, SpectralClustering
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import kneighbors_graph

from sklearn.cluster import estimate_bandwidth
from sklearn.metrics import silhouette_score

In [ ]:
def make_datasets(seed=0, n=800):
    rng = np.random.RandomState(seed)

    # 1) globular blobs (KMeans-friendly)
    X1, _ = make_blobs(n_samples=n, centers=4, cluster_std=0.8, random_state=seed)


    # 2) moons (non-convex)
    X2, _ = make_moons(n_samples=n, noise=0.10, random_state=seed)

    # 3) circles (non-convex)
    X3, _ = make_circles(n_samples=n, noise=0.06, factor=0.45, random_state=seed)

    # 4) different densities (KMeans/GMM can struggle)
    X4a, _ = make_blobs(n_samples=n//2, centers=[(-2, 0)], cluster_std=0.35, random_state=seed)
    X4b, _ = make_blobs(n_samples=n//2, centers=[( 2, 0)], cluster_std=1.30, random_state=seed+1)
    X4 = np.vstack([X4a, X4b])

    return [
        ("Blobs", X1),
        ("Moons", X2),
        ("Circles", X3),
        ("Unequal density", X4),
    ]

def plot_clusters(ax, X, labels, title=""):
    # labels: -1 often means noise (DBSCAN/OPTICS)
    unique = np.unique(labels)
    ax.scatter(X[:, 0], X[:, 1], c=labels, s=14, edgecolor="k", linewidth=0.2)
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])

def safe_silhouette(X, labels):
    # remove noise points for silhouette
    mask = labels != -1
    labs = labels[mask]
    if mask.sum() < 10:
        return None
    if len(np.unique(labs)) < 2:
        return None
    return silhouette_score(X[mask], labs)


def get_clusterers(X, k_true=2, conn_n_neighbors=5, seed=0):
    # A connectivity graph for hierarchical clustering
    conn = kneighbors_graph(X, n_neighbors=conn_n_neighbors, include_self=False)

    clusterers = [
        ("KMeans", KMeans(n_clusters=k_true, n_init=10, random_state=seed)),
        ("GMM", GaussianMixture(n_components=k_true, covariance_type="full", random_state=seed)),
        ("Agglo (comp)", AgglomerativeClustering(n_clusters=k_true, linkage="complete")),
        ("Agglo (comp)+conn", AgglomerativeClustering(n_clusters=k_true, linkage="complete", connectivity=conn)),
    ]

    return clusterers

In [ ]:
datasets = make_datasets(seed=0, n=900)

fig_rows = len(datasets)
# pick how many models you want to show (trim list if too wide)
show_models = None  # None = show all in get_clusterers()

for (dname, X_raw) in datasets:
    # scale helps distance-based clustering (KMeans, spectral, hierarchical, DBSCAN)
    X = StandardScaler().fit_transform(X_raw)

    # choose k for algorithms that need it (blobs: 4, anisotropic: 3, moons/circles: 2, unequal density: 2)
    k = {"Blobs": 4, "Anisotropic": 3, "Moons": 2, "Circles": 2, "Unequal density": 2}[dname]

    clusterers = get_clusterers(X, k_true=k, seed=0)
    if show_models is not None:
        clusterers = [c for c in clusterers if c[0] in show_models]

    fig, axes = plt.subplots(1, len(clusterers), figsize=(4.2 * len(clusterers), 3.6))
    if len(clusterers) == 1:
        axes = [axes]

    for ax, (mname, model) in zip(axes, clusterers):
        # Fit + get labels
        if isinstance(model, GaussianMixture):
            model.fit(X)
            labels = model.predict(X)
        else:
            labels = model.fit_predict(X)

        sil = safe_silhouette(X, labels)
        sil_txt = f"sil={sil:.2f}" if sil is not None else "sil=NA"
        plot_clusters(ax, X, labels, title=f"{dname}\n{mname}\n{sil_txt}")

    plt.tight_layout()
    plt.show()

In [ ]:
import networkx as nx

# moons (non-convex)
n = 500
X, _ = make_moons(n_samples=n, noise=0.02, random_state=382)

n = 5
conn = kneighbors_graph(X, n_neighbors=n, include_self=False)
G = nx.from_scipy_sparse_array(conn)

plt.figure(figsize=(5,5))
plt.scatter(X[:,0], X[:,1], s=10)
for i, j in zip(*conn.nonzero()):
    plt.plot([X[i,0], X[j,0]], [X[i,1], X[j,1]], color='gray', alpha=0.1)
plt.title(f"kNN Connectivity Graph (n_neighbors={n})")
plt.xticks([]); plt.yticks([])
plt.show()

In [ ]:
# connectivity analysis - local geometry.

# moons (non-convex)
n = 500
X, _ = make_moons(n_samples=n, noise=0.02, random_state=382)

# Try different connectivity strengths
neighbor_values = [1, 3, 5, 10, 20, 50]

fig, axes = plt.subplots(1, len(neighbor_values), figsize=(4*len(neighbor_values), 3.5))

for ax, n in zip(axes, neighbor_values):

    # Build kNN graph
    conn = kneighbors_graph(X, n_neighbors=n, include_self=False)

    model = AgglomerativeClustering(
        n_clusters=2,
        linkage="complete",
        connectivity=conn,
    )

    labels = model.fit_predict(X)

    ax.scatter(X[:, 0], X[:, 1], c=labels, s=12)
    ax.set_title(f"n_neighbors = {n}")
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle("Agglomerative Clustering with Different Connectivity Graphs", y=1.02);
plt.tight_layout();
plt.show();

In [ ]:
import networkx as nx

# circles (non-convex)
n = 400
X, _ = make_circles(n_samples=n, noise=0.06, factor=0.45, random_state=39)


n = 4
conn = kneighbors_graph(X, n_neighbors=n, include_self=False)
G = nx.from_scipy_sparse_array(conn)

plt.figure(figsize=(5,5))
plt.scatter(X[:,0], X[:,1], s=10)
for i, j in zip(*conn.nonzero()):
    plt.plot([X[i,0], X[j,0]], [X[i,1], X[j,1]], color='gray', alpha=0.1)
plt.title(f"kNN Connectivity Graph (n_neighbors={n})")
plt.xticks([]); plt.yticks([])
plt.show()

In [ ]:
# connectivity analysis - local geometry.

# circles (non-convex)
n = 400
X, _ = make_circles(n_samples=n, noise=0.06, factor=0.45, random_state=39)

# Try different connectivity strengths
neighbor_values = [1, 3, 5, 10, 20, 50]

fig, axes = plt.subplots(1, len(neighbor_values), figsize=(4*len(neighbor_values), 3.5))

for ax, n in zip(axes, neighbor_values):

    # Build kNN graph
    conn = kneighbors_graph(X, n_neighbors=n, include_self=False)

    model = AgglomerativeClustering(
        n_clusters=2,
        linkage="complete",
        connectivity=conn,
    )

    labels = model.fit_predict(X)

    ax.scatter(X[:, 0], X[:, 1], c=labels, s=12)
    ax.set_title(f"n_neighbors = {n}")
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle("Agglomerative Clustering with Different Connectivity Graphs", y=1.02);
plt.tight_layout();
plt.show();

In [ ]:
# spectral clustering (kmeans, nn, versus rbf)

from sklearn.cluster import SpectralClustering

X, _ = make_circles(n_samples=900, noise=0.06, factor=0.45)
X = StandardScaler().fit_transform(X)
k = 2

def plot_labels(ax, X, labels, title):
    ax.scatter(X[:, 0], X[:, 1], c=labels, s=14, edgecolor="k", linewidth=0.2)
    ax.set_title(title)
    ax.set_xticks([]); ax.set_yticks([])

spec_nn = SpectralClustering(
    n_clusters=k, affinity="nearest_neighbors", n_neighbors=10,
    assign_labels="kmeans", random_state=0
).fit(X)

# RBF: gamma controls similarity decay; try a few values
gammas = [0.5, 1.0, 5.0]

fig, axes = plt.subplots(1, 1 + len(gammas), figsize=(4.2*(1+len(gammas)), 3.6))
plot_labels(axes[0], X, spec_nn.labels_, "Spectral\nnearest_neighbors")

for ax, g in zip(axes[1:], gammas):
    spec_rbf = SpectralClustering(
        n_clusters=k, affinity="rbf", gamma=g,
        assign_labels="kmeans", random_state=0
    ).fit(X)
    plot_labels(ax, X, spec_rbf.labels_, f"Spectral\nrbf, gamma={g}")

plt.tight_layout()
plt.show()

In [ ]:
# spectral clustering

# moons (non-convex)
n = 500
X, _ = make_moons(n_samples=n, noise=0.05, random_state=382)
X = StandardScaler().fit_transform(X)
k = 2

spec_nn = SpectralClustering(
    n_clusters=k, affinity="nearest_neighbors", n_neighbors=10,
    assign_labels="kmeans", random_state=0
).fit(X)

# RBF: gamma controls similarity decay; try a few values
gammas = [0.5, 1.0, 5.0]

fig, axes = plt.subplots(1, 1 + len(gammas), figsize=(4.2*(1+len(gammas)), 3.6))
plot_labels(axes[0], X, spec_nn.labels_, "Spectral\nnearest_neighbors")

for ax, g in zip(axes[1:], gammas):
    spec_rbf = SpectralClustering(
        n_clusters=k, affinity="rbf", gamma=g,
        assign_labels="kmeans", random_state=0
    ).fit(X)
    plot_labels(ax, X, spec_rbf.labels_, f"Spectral\nrbf, gamma={g}")

plt.tight_layout()
plt.show()

In [ ]:
### how many clusters?

gt_cluster = 5
n = 500
# moons (non-convex)
X, y = make_blobs(n_samples=n, n_features=2, centers=gt_cluster, cluster_std=0.5, random_state=0)
X = StandardScaler().fit_transform(X)

plt.figure()
plt.scatter(X[:, 0], X[:, 1], s=14, edgecolor="k", linewidth=0.2)
ax.set_xticks([])
ax.set_yticks([])
plt.tight_layout()
plt.show()


def stability_kmeans(X, k, B=20):
    n = len(X)
    labels_list = []
    idx_list = []

    for b in range(B):
        idx = np.random.choice(n, int(0.8*n), replace=True)
        Xb = X[idx]
        labels = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(Xb)
        labels_list.append(labels)
        idx_list.append(idx)

    aris = []
    for i in range(B):
        for j in range(i+1, B):
            common = np.intersect1d(idx_list[i], idx_list[j])
            if len(common) < 20:
                continue
            pos_i = [np.where(idx_list[i]==c)[0][0] for c in common]
            pos_j = [np.where(idx_list[j]==c)[0][0] for c in common]
            aris.append(adjusted_rand_score(labels_list[i][pos_i],
                                            labels_list[j][pos_j]))
    return np.mean(aris)

ks = range(2,8)

sil_scores = []
inertias = []
stabilities = []
bics = []

for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(X)
    labels = km.labels_

    sil_scores.append(silhouette_score(X, labels))
    inertias.append(km.inertia_)
    stabilities.append(stability_kmeans(X, k))

# Plot
plt.figure(figsize=(15,4))

plt.subplot(1,3,1)
plt.plot(ks, sil_scores, marker='o')
plt.title("Silhouette")

plt.subplot(1,3,2)
plt.plot(ks, inertias, marker='o')
plt.title("Inertia (KMeans)")

plt.subplot(1,3,3)
plt.plot(ks, stabilities, marker='o')
plt.title("Stability (ARI)")


plt.tight_layout()
plt.show()